# Telemetry Analysis

Analyse des données de télémétrie machines. Chargement du CSV, puis génération des distributions par machine et par métrique.

## Plan du notebook

1. **Chargement des données** — lecture du CSV, typage des colonnes, définition des métriques et des machines
2. **Anonymisation** — hachage des données personnelles du fichier incidents (operator_name, operator_badge, incident_id)
3. **Statistiques descriptives** — moyenne et écart type par métrique
4. **Valeurs manquantes** — visualisation du nombre de nulls par machine et par métrique
5. **Imputation** — remplacement des valeurs manquantes par la moyenne de chaque métrique
6. **Distribution globale** — histogrammes par métrique sur l'ensemble des machines
7. **Valeurs atypiques** — identification des valeurs avec un Z-score > 3 par métrique
8. **Export** — export des données anonymisées et de la télémétrie dans `ingestion/silver/`

## Chargement des données

Import des librairies et lecture du fichier `telemetry.csv`. Les colonnes numériques sont castées en float et `METRICS` / `machines` sont définis ici pour être réutilisés dans les cellules suivantes.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

df = pd.read_csv("artifacts/ingestions/incidents/telemetry.csv")
df = df[df["machine_id"] != "machine_id"]  # drop duplicate headers

for col in ["temperature_c", "pressure_bar", "voltage_mean_v", "rotation_mean_rpm", "pieces_produced"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.rename(columns={
    "temperature_c": "temperature_mean",
    "pressure_bar": "pressure_mean",
    "voltage_mean_v": "voltage_mean",
    "rotation_mean_rpm": "rotation_mean",
})

METRICS = {
    "temperature": "temperature_mean",
    "pressure": "pressure_mean",
    "voltage": "voltage_mean",
    "rotation": "rotation_mean",
}

machines = sorted(df["machine_id"].unique())
print(f"Shape: {df.shape} | Machines: {len(machines)}")
df.head()

Shape: (135626, 7) | Machines: 15


,machine_id,timestamp,temperature_mean,pressure_mean,voltage_mean,rotation_mean,pieces_produced
0,MACH-01,2025-06-01 00:00:00,46.348,198.203,227.568,1541.787,4
1,MACH-01,2025-06-01 00:00:00,46.332,198.206,227.570,1541.760,4
2,MACH-01,2025-06-01 01:00:00,48.762,198.295,227.480,1537.860,4
3,MACH-01,2025-06-01 02:00:00,51.352,199.545,228.680,1584.660,13
4,MACH-01,2025-06-01 03:00:00,49.512,201.641,228.440,1588.960,10


## Anonymisation des données

Le fichier `releves_incidents.csv` contient des données personnelles (noms et badges des opérateurs). Ces données sont remplacées par des identifiants anonymes avant toute analyse :
- `operator_name` → haché en `operator_id`
- `operator_badge` → haché en `badge_id`
- `incident_id` → haché pour éviter toute traçabilité directe

In [98]:
import hashlib

INCIDENTS_CSV = "artifacts/ingestions/incidents/releves_incidents.csv"

df_incidents = pd.read_csv(INCIDENTS_CSV)

def hash_cell(value, idx, col):
    return hashlib.sha256(f"{col}:{idx}:{value}".encode()).hexdigest()[:12]

df_incidents["incident_id"]    = [hash_cell(v, i, "incident_id")    for i, v in enumerate(df_incidents["incident_id"])]
df_incidents["operator_name"]  = [hash_cell(v, i, "operator_name")  for i, v in enumerate(df_incidents["operator_name"])]
df_incidents["operator_badge"] = [hash_cell(v, i, "operator_badge") for i, v in enumerate(df_incidents["operator_badge"])]

df_incidents = df_incidents.rename(columns={"operator_name": "operator_id", "operator_badge": "badge_id"})

print(f"Anonymisation terminée : {len(df_incidents)} lignes traitées")
df_incidents.head()

Anonymisation terminée : 1245 lignes traitées


,incident_id,date,time,operator_id,machine_id,severity,badge_id,comment,shift,type_surchauffe,type_baisse_pression,type_vibration,type_bruit_mecanique,type_surconsommation,type_blocage_mecanique,type_alarme_capteur,type_arret_urgence,type_defaut_qualite
0,35d6703be341,2025-06-01,05:42,e365a3dff667,MACH-06,4,5cb39ae52ff1,chauffe anormale,nuit,1,0,0,0,0,0,0,0,0
1,09d756e0caa1,2025-06-01,21:08,6e43a61e417f,MACH-15,3,c51cf705adbc,micro-fuite / baisse pression,apres-midi,0,1,0,0,0,0,0,0,0
2,da636a4c0266,2025-06-02,05:43,7711c93c3fd6,MACH-10,2,634e53074598,défaut capteur confirmé,nuit,0,0,0,0,0,0,1,0,0
3,ad1bb0825961,2025-06-03,05:43,9a42e2288eeb,MACH-10,3,c108927d92d2,signal capteur hors plage,nuit,0,0,0,0,0,0,1,0,0
4,3034a16bc6ca,2025-06-04,01:01,be2f0a653ff7,MACH-14,2,c80e6393c2ca,alerte température haute,nuit,1,0,0,0,0,0,0,0,0


## Statistiques descriptives

Moyenne et écart type de chaque métrique sur l'ensemble du jeu de données.

In [99]:
stats = df[[col for col in METRICS.values()]].agg(["mean", "std"]).rename(index={"mean": "Moyenne", "std": "Écart type"})
stats.columns = METRICS.keys()
stats.round(2)

,temperature,pressure,voltage,rotation
Moyenne,48.18,199.77,227.63,1589.20
Écart type,5.25,2.37,2.30,45.95


## Valeurs manquantes par machine

Pour chaque métrique, affiche le nombre de valeurs nulles par machine sous forme de graphique en barres groupées.

In [100]:
MISSING_DIR = Path("artifacts/telemetry/missing_values")
MISSING_DIR.mkdir(parents=True, exist_ok=True)

null_counts = (
    df.groupby("machine_id")[[col for col in METRICS.values()]]
    .apply(lambda g: g.isnull().sum())
    .rename(columns={v: k for k, v in METRICS.items()})
)

display(null_counts)

# Graph global : barres groupées par métrique pour chaque machine
fig, ax = plt.subplots(figsize=(14, 6))
null_counts.plot(kind="bar", ax=ax)
ax.set_title("Valeurs manquantes par machine et par métrique")
ax.set_xlabel("Machine")
ax.set_ylabel("Nombre de valeurs manquantes")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(title="Métrique")
plt.tight_layout()
fig.savefig(MISSING_DIR / "missing_values_global.png", dpi=100)
plt.close(fig)

# Un graph par métrique
for metric_name in METRICS:
    fig, ax = plt.subplots(figsize=(12, 5))
    null_counts[metric_name].plot(kind="bar", ax=ax, color="darkorange")
    ax.set_title(f"Valeurs manquantes par machine — {metric_name}")
    ax.set_xlabel("Machine")
    ax.set_ylabel("Nombre de valeurs manquantes")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(MISSING_DIR / f"{metric_name}_missing_values.png", dpi=100)
    plt.close(fig)

print(f"Done. {1 + len(METRICS)} graphs saved to {MISSING_DIR}")

,temperature,pressure,voltage,rotation
machine_id,,,,
MACH-01,44,66,0,77
MACH-02,75,43,0,53
MACH-03,55,52,0,80
MACH-04,49,53,0,105
MACH-05,62,56,0,54
MACH-06,43,72,0,71
MACH-07,51,92,0,63
MACH-08,76,29,0,69
MACH-09,65,114,0,39


Done. 5 graphs saved to artifacts/telemetry/missing_values


## Imputation des valeurs manquantes

Les valeurs manquantes sont remplacées par la moyenne de leur métrique respective.

In [101]:
missing_before = df[list(METRICS.values())].isnull().sum().sum()

for metric_name, col in METRICS.items():
    mean_val = round(df[col].mean(), 2)
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        print(f"{metric_name} : valeur moyenne utilisée = {mean_val} ({n_missing} valeur(s) imputée(s))")
    df[col] = df[col].fillna(mean_val)

missing_after = df[list(METRICS.values())].isnull().sum().sum()

print(f"\nValeurs manquantes détectées  : {missing_before}")
print(f"Valeurs manquantes restantes  : {missing_after}")

temperature : valeur moyenne utilisée = 48.18 (894 valeur(s) imputée(s))
pressure : valeur moyenne utilisée = 199.77 (995 valeur(s) imputée(s))
rotation : valeur moyenne utilisée = 1589.2 (958 valeur(s) imputée(s))

Valeurs manquantes détectées  : 2847
Valeurs manquantes restantes  : 0


## Distribution globale par métrique

Distribution de chaque métrique sur l'ensemble des machines, sans distinction. Les graphiques sont exportés dans `artifacts/telemetry/metrics/`.

In [102]:
METRICS_OUTPUT_DIR = Path("artifacts/telemetry/metrics")
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for metric_name, col in METRICS.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[col].dropna().astype(float), kde=True, ax=ax, color="darkorange")
    ax.set_title(f"{metric_name} — global distribution (all machines)")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    plt.tight_layout()
    filename = f"{metric_name}_DISTRIBUTION.png"
    fig.savefig(METRICS_OUTPUT_DIR / filename, dpi=100)
    plt.close(fig)

print(f"Done. {len(METRICS)} graphs saved to {METRICS_OUTPUT_DIR}")

Done. 4 graphs saved to artifacts/telemetry/metrics


## Valeurs atypiques (Z-score > 3)

Pour chaque métrique, liste des lignes dont la valeur s'écarte de plus de 3 écarts types de la moyenne.

In [103]:
from scipy import stats

for metric_name, col in METRICS.items():
    series = df[col].dropna()
    z_scores = stats.zscore(series)
    mask = abs(z_scores) > 3
    outliers = df.loc[series.index[mask], ["machine_id", "timestamp", col]]
    total = len(series)
    n_outliers = len(outliers)
    pct = n_outliers / total * 100
    val_min, val_max = outliers[col].min(), outliers[col].max()
    print(f"\n{metric_name} — {n_outliers} valeur(s) atypique(s) / {total} total ({pct:.2f}%) | min: {val_min:.2f}, max: {val_max:.2f}")
    if not outliers.empty:
        display(outliers.reset_index(drop=True))


temperature — 275 valeur(s) atypique(s) / 135626 total (0.20%) | min: 32.00, max: 80.00


,machine_id,timestamp,temperature_mean
0,MACH-01,2025-06-15 23:00:00,65.114
1,MACH-01,2025-06-16 00:00:00,70.133
2,MACH-01,2025-06-16 17:00:00,67.146
3,MACH-01,2025-06-16 18:00:00,69.536
4,MACH-01,2025-06-16 19:00:00,72.378
...,...,...,...
270,MACH-15,2025-07-27 14:00:00,71.878
271,MACH-15,2025-07-27 14:00:00,71.862
272,MACH-15,2025-07-27 15:00:00,75.874
273,MACH-15,2025-07-27 16:00:00,80.000



pressure — 1061 valeur(s) atypique(s) / 135626 total (0.78%) | min: 159.98, max: 215.81


,machine_id,timestamp,pressure_mean
0,MACH-01,2025-06-16 22:00:00,191.276
1,MACH-01,2025-06-16 23:00:00,189.239
2,MACH-01,2025-06-17 00:00:00,186.456
3,MACH-01,2025-08-15 16:00:00,190.200
4,MACH-01,2025-08-15 17:00:00,190.222
...,...,...,...
1056,MACH-15,2026-05-03 16:00:00,189.212
1057,MACH-15,2026-05-03 17:00:00,184.882
1058,MACH-15,2026-05-03 18:00:00,186.012
1059,MACH-15,2026-05-03 19:00:00,183.546



voltage — 365 valeur(s) atypique(s) / 135626 total (0.27%) | min: 234.56, max: 242.00


,machine_id,timestamp,voltage_mean
0,MACH-01,2025-06-16 23:00:00,235.964
1,MACH-01,2025-06-17 00:00:00,237.264
2,MACH-01,2025-10-04 14:00:00,235.624
3,MACH-01,2025-10-04 15:00:00,236.743
4,MACH-01,2025-10-04 16:00:00,238.546
...,...,...,...
360,MACH-15,2026-04-08 06:00:00,235.970
361,MACH-15,2026-04-08 08:00:00,235.340
362,MACH-15,2026-04-08 09:00:00,235.370
363,MACH-15,2026-04-08 10:00:00,235.050



rotation — 629 valeur(s) atypique(s) / 135626 total (0.46%) | min: 1100.00, max: 1900.00


,machine_id,timestamp,rotation_mean
0,MACH-01,2025-06-12 15:00:00,1746.542
1,MACH-01,2025-06-12 18:00:00,1900.000
2,MACH-01,2025-06-13 06:00:00,1788.254
3,MACH-01,2025-06-13 07:00:00,1900.000
4,MACH-01,2025-06-13 08:00:00,1900.000
...,...,...,...
624,MACH-15,2026-05-03 22:00:00,1430.816
625,MACH-15,2026-05-03 22:00:00,1430.821
626,MACH-15,2026-05-03 23:00:00,1362.084
627,MACH-15,2026-05-04 00:00:00,1445.210


## Export des données

Export du fichier incidents anonymisé vers la couche silver et de la télémétrie vers la couche gold.

In [104]:
SILVER_PATH = Path("artifacts/ingestions/silver")
SILVER_PATH.mkdir(parents=True, exist_ok=True)

df_incidents.to_csv(SILVER_PATH / "silver_incidents.csv", index=False)
print(f"Incidents anonymisés exportés : {SILVER_PATH / 'silver_incidents.csv'}")

df.to_csv(SILVER_PATH / "silver_telemetry.csv", index=False)
print(f"Télémétrie exportée           : {SILVER_PATH / 'silver_telemetry.csv'}")

Incidents anonymisés exportés : artifacts/ingestions/silver/silver_incidents.csv
Télémétrie exportée           : artifacts/ingestions/silver/silver_telemetry.csv
